<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 37px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  Machine Learning<br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  -<br>
  </div> 
  
<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      Illustration with Random Forests<br><br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 25px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      <font color=orange> 2 - Data preprocessing </font>
  </div>

<a id="plan"></a>

## Data Preprocessing

-  [About the dataset and Kaggle](#about)<br>
-  [Load the data](#load_data)<br>
-  [Look at the data](#look_data)<br>
-  [Look at the columns](#look_columns)<br>
-  [Variable to predict](#output_variable)<br>
-  [Naive first model training](#naive_training)<br>
-  [Convert date times](#convert_date)<br>
-  [Convert strings into numerotated categories](#convert_strings)<br>
-  [Inspect missing values](#nans)<br>
-  [Save preprocessed data](#save_data)<br>
-  [Fully numericalize data](#numericalize_data)<br>



### Imports

In [ ]:
# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [ ]:
# base modules
from pathlib import Path

# for manipulating data
import numpy as np
import pandas as pd

# for Machine Learning
from sklearn.ensemble import RandomForestRegressor

In [ ]:
from eclyon.transforms import (
    add_date_columns,
    change_columns_from_str_to_categorical,
    process_df,
)

# Data preprocessing

[Back to top](#plan)

### $\bullet$ About the dataset and Kaggle

[Back to top](#plan)

We will be looking at the Blue Book for Bulldozers Kaggle Competition: "The goal of the contest is to predict the sale price of a particular piece of heavy equiment at auction based on it's usage, equipment type, and configuration.  The data is sourced from auction result postings and includes information on usage and equipment configurations."

This is a very common type of dataset and prediciton problem, and similar to what you may see in your project or workplace.

Kaggle is an awesome resource for aspiring data scientists or anyone looking to improve their machine learning skills.  There is nothing like being able to get hands-on practice and receiving real-time feedback to help you improve your skills.

Kaggle provides:

1. Interesting data sets
2. Feedback on how you're doing
3. A leader board to see what's good, what's possible, and what's state-of-art.
4. Blog posts by winning contestants share useful tips and techniques.



Kaggle provides info about some of the fields of our dataset; on the [Kaggle Data info](https://www.kaggle.com/c/bluebook-for-bulldozers/data) page they say the following:

For this competition, you are predicting the sale price of bulldozers sold at auctions. The data for this competition is split into three parts:

- **Train.csv** is the training set, which contains data through the end of 2011.
- **Valid.csv** is the validation set, which contains data from January 1, 2012 - April 30, 2012. You make predictions on this set throughout the majority of the competition. Your score on this set is used to create the public leaderboard.
- **Test.csv** is the test set, which won't be released until the last week of the competition. It contains data from May 1, 2012 - November 2012. Your score on the test set determines your final rank for the competition.

### $\bullet$ Import the data

[Back to top](#plan)

In [ ]:
path_to_repo = Path("..").resolve()
path_to_data = path_to_repo / "data"

In [ ]:
df_raw = pd.read_csv(path_to_data / "Train.csv", low_memory=False, parse_dates=["saledate"])

In [ ]:
df_raw = df_raw.convert_dtypes()

### $\bullet$ Look at the data

[Back to top](#plan)

In any sort of data science work, it's **important to look at your data**, to make sure you understand the format, how it's stored, what type of values it holds, etc.<br> 
Even if you've read descriptions about your data, the actual data may not be what you expect.

The key fields that are in train.csv are:

- SalesID: the unique identifier of the sale
- MachineID: the unique identifier of a machine.  A machine can be sold multiple times
- saleprice: what the machine sold for at auction (only provided in train.csv)
- saledate: the date of the sale

In [ ]:
# print the 5 first and last rows
df_raw

In [ ]:
# print the first n rows (default is n = 5)
# df_raw.head(n = 3)
df_raw.head(3)

In [ ]:
help(df_raw.head)

### $\bullet$ Look at the columns

[Back to top](#plan)

In [ ]:
df_raw.columns

In [ ]:
len(df_raw.columns)

In [ ]:
def display_all(df):
    with pd.option_context("display.max_rows", 1000, "display.max_columns", 1000):
        display(df)

In [ ]:
df_raw.shape

In [ ]:
display_all(df_raw.head().T)

In [ ]:
display_all(df_raw.describe(include="all").T)

### $\bullet$ Variable to predict

[Back to top](#plan)

The variable we want to predict is the sale price of a bulldozer, based on its description. This means that we seek to **predict** values within the _SalePrice_ column of the dataset, by using the other columns as **expainatory variables** for prediction.

It's important to note what metric is being used for a project. Generally, selecting the metric(s) is an important part of the project setup. However, in this case Kaggle tells us what metric to use: RMSLE (root mean squared log error) between the actual and predicted auction prices. Therefore we take the log of the prices, so that RMSE will give us what we need.

In [ ]:
df_raw.SalePrice

In [ ]:
sale_prices = df_raw.SalePrice.tolist()

In [ ]:
df_raw.SalePrice = np.log(df_raw.SalePrice)

In [ ]:
df_raw.SalePrice

In [ ]:
display_all(df_raw.describe(include="all").T)

In [ ]:
# same as
# df_raw.SalePrice
df_raw["SalePrice"]

In [ ]:
# to convert back prices to original values
np.exp(df_raw.SalePrice)

### $\bullet$ Naive first model training

[Back to top](#plan)

In [ ]:
model = RandomForestRegressor(n_jobs=-1)

In [ ]:
# The following code is supposed to fail due to string values in the input data
model.fit(df_raw.drop("SalePrice", axis=1), df_raw.SalePrice)

This dataset contains a mix of **continuous** and **categorical** variables.

The following method extracts particular date fields from a complete datetime for the purpose of constructing categoricals.  You should always consider this feature extraction step when working with date-time. Without expanding your date-time into these additional fields, you can't capture any trend/cyclical behavior as a function of time at any of these granularities.

### $\bullet$ Convert date times

[Back to top](#plan)

In [ ]:
df_raw.fiModelDesc.value_counts()

In [ ]:
df_raw.saledate

In [ ]:
# date-time object was created within the df with parse_dates=["saledate"]
df_raw.saledate.dt

In [ ]:
# add_datepart adds new columns to the dataframe
df_transformed = add_date_columns(df_raw, fields=["saledate"])

In [ ]:
# comparaison of dataframe shape before & after applying add_datepart
print(df_raw.shape)
print(df_transformed.shape)

In [ ]:
df_transformed.saleYear.head()

In [ ]:
df_transformed.columns

In [ ]:
df_transformed.saleDayofweek

### $\bullet$ Convert strings into numerotated categories

[Back to top](#plan)

The categorical variables are currently stored as strings, which is inefficient, and doesn't provide the numeric coding required for a random forest. Therefore we convert strings to pandas categories.

In [ ]:
df_transformed = change_columns_from_str_to_categorical(df_transformed)

We can specify the order to use for categorical variables if we wish:

In [ ]:
df_transformed.UsageBand

In [ ]:
df_transformed.UsageBand.cat.categories.tolist()

In [ ]:
df_transformed.UsageBand.dtype

In [ ]:
df_transformed.UsageBand = df_transformed.UsageBand.cat.set_categories(["Low", "Medium", "High"], ordered=True)

In [ ]:
df_transformed.UsageBand.cat.categories.tolist()

Normally, pandas will continue displaying the text categories, while treating them as numerical data internally. Optionally, we can replace the text categories with numbers, which will make this variable non-categorical, like so:.

In [ ]:
df_transformed.UsageBand

In [ ]:
df_transformed.UsageBand.cat.categories

In [ ]:
df_transformed.UsageBand.cat.codes

In [ ]:
# This step overwrites the raw categories with integers
df_transformed.UsageBand = df_transformed.UsageBand.cat.codes

In [ ]:
# - values corresponds to NaN values in the original column
df_transformed.UsageBand

### $\bullet$ Inspect missing values

[Back to top](#plan)

We're still not quite done - for instance we have lots of missing values, which we can't pass directly to a random forest.

In [ ]:
df_transformed.isnull().describe()

In [ ]:
display_all(df_transformed.isnull().describe().T)

In [ ]:
# display amount of missing values per column :
# column name              ratio of nans
display_all(df_transformed.isnull().sum().sort_index() / len(df_transformed))

### $\bullet$ Save preprocessed data

[Back to top](#plan)

But let's save this file for now, since it's already in format can we be stored and accessed efficiently.

In [ ]:
# 18 MB
df_transformed.to_feather(path_to_data / "transformed")

In [ ]:
# 140 MB !!
df_transformed.to_csv(path_to_data / "transformed.csv", index=False)

### $\bullet$ Fully numericalize data

[Back to top](#plan)

We'll replace categories with their numeric codes, handle missing continuous values, and split the dependent variable into a separate variable.

In [ ]:
df_transformed = pd.read_feather(path_to_data / "transformed")

In [ ]:
df_processed, y, nas = process_df(df_transformed, y_field="SalePrice")

In [ ]:
df_processed.columns

In [ ]:
df_processed.head()

In [ ]:
y

In [ ]:
nas

We now have something we can pass to a random forest!

[Back to top](#plan)